# AI Bootcamp: Complete LLM & RAG System
## From Theory to Production in 2 Hours

This notebook demonstrates:
- LLM basics and prompt engineering
- Text splitting and chunking strategies
- Vector embeddings and similarity search
- RAG system implementation
- LangChain framework usage
- AI agents with tools
- Final integrated system

## Setup and Dependencies

In [2]:
# # Install required packages (run once)
# !pip install langchain langchain-community langchain-openai
# !pip install chromadb sentence-transformers
# !pip install pypdf python-docx
# !pip install numpy pandas matplotlib plotly

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
from typing import List, Dict, Any
import json
from pathlib import Path

# LangChain imports
from langchain.llms import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import (
    CharacterTextSplitter, 
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
from langchain.document_loaders import TextLoader, PyPDFLoader
from langchain.chains import RetrievalQA, ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, Tool, AgentType
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import Document

# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = ''  # Replace with your actual key

print("✅ All dependencies imported successfully!")

## Module 1: LLM Basics (20 minutes)
### Understanding Large Language Models

In [6]:
# Initialize LLM
llm = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")

# Basic LLM interaction
response = llm.predict("Explain what a Large Language Model is in simple terms")
print("🤖 LLM Response:")
print(response)

🤖 LLM Response:
A Large Language Model is a type of computer program that is designed to understand and generate human language. It uses a large amount of text data to learn how to write and speak like a human. These models can be used for tasks like answering questions, writing articles, or even carrying on a conversation. They are able to understand the context of a sentence and generate text that is coherent and meaningful.


In [7]:
# Prompt Engineering Examples
prompts = {
    "Basic": "What is AI?",
    "Detailed": "Explain artificial intelligence to a 10-year-old with examples",
    "Structured": """Explain AI in the following format:
            Definition: [brief definition]
            Examples: [3 real-world examples]
            Impact: [how it affects daily life]""",
    "Role-based": "You are a friendly AI teacher. Explain machine learning to a curious student."
}

print("📝 Prompt Engineering Comparison:")
for prompt_type, prompt in prompts.items():
    print(f"\n--- {prompt_type.upper()} PROMPT ---")
    print(f"Input: {prompt}")
    response = llm.predict(prompt)
    print(f"Output: {response[:200]}...")
    print("-" * 50)

📝 Prompt Engineering Comparison:

--- BASIC PROMPT ---
Input: What is AI?
Output: AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, problem-solving, percept...
--------------------------------------------------

--- DETAILED PROMPT ---
Input: Explain artificial intelligence to a 10-year-old with examples
Output: Artificial intelligence (AI) is like giving a computer the ability to think and learn like a human. Just like you learn new things and get better at them over time, AI can do the same. 

For example, ...
--------------------------------------------------

--- STRUCTURED PROMPT ---
Input: Explain AI in the following format:
    Definition: [brief definition]
    Examples: [3 real-world examples]
    Impact: [how it affects daily life]
Output: Definition: AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, 

## Module 2: Text Splitting & Chunking (15 minutes)
### Why Text Splitting Matters

In [8]:
# Sample document for demonstration
sample_text = """
Artificial Intelligence (AI) is a rapidly evolving field that encompasses machine learning, 
natural language processing, computer vision, and robotics. Machine learning is a subset of AI 
that focuses on algorithms that can learn from data without being explicitly programmed.

Deep learning, a subset of machine learning, uses neural networks with multiple layers to 
process and analyze complex data patterns. These networks are inspired by the human brain's 
structure and function.

Large Language Models (LLMs) like GPT represent a significant breakthrough in natural language 
processing. They are trained on vast amounts of text data and can generate human-like responses 
to prompts and questions.

Retrieval-Augmented Generation (RAG) combines the power of LLMs with external knowledge sources 
to provide more accurate and up-to-date information. This approach helps reduce hallucinations 
and improves the reliability of AI-generated content.
"""

print(f"📄 Sample Document Length: {len(sample_text)} characters")
print(f"📄 Sample Document:\n{sample_text[:200]}...")

📄 Sample Document Length: 956 characters
📄 Sample Document:

Artificial Intelligence (AI) is a rapidly evolving field that encompasses machine learning, 
natural language processing, computer vision, and robotics. Machine learning is a subset of AI 
that focus...


In [9]:
# Different Text Splitting Strategies

# 1. Character-based splitting
char_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separator="\n\n"
)
char_chunks = char_splitter.split_text(sample_text)

# 2. Recursive character splitting (recommended)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
recursive_chunks = recursive_splitter.split_text(sample_text)

# 3. Token-based splitting
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
token_chunks = token_splitter.split_text(sample_text)

# Compare results
print("🔪 Text Splitting Comparison:")
print(f"Character-based chunks: {len(char_chunks)}")
print(f"Recursive chunks: {len(recursive_chunks)}")
print(f"Token-based chunks: {len(token_chunks)}")

print("\n📋 Sample Recursive Chunks:")
for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i+1}: {chunk.strip()[:100]}...")
    print(f"Length: {len(chunk)} characters\n")

Created a chunk of size 278, which is longer than the specified 200
Created a chunk of size 207, which is longer than the specified 200
Created a chunk of size 218, which is longer than the specified 200


🔪 Text Splitting Comparison:
Character-based chunks: 4
Recursive chunks: 4
Token-based chunks: 5

📋 Sample Recursive Chunks:
Chunk 1: Artificial Intelligence (AI) is a rapidly evolving field that encompasses machine learning, 
natural...
Length: 277 characters

Chunk 2: Deep learning, a subset of machine learning, uses neural networks with multiple layers to 
process a...
Length: 207 characters

Chunk 3: Large Language Models (LLMs) like GPT represent a significant breakthrough in natural language 
proc...
Length: 218 characters

Chunk 4: Retrieval-Augmented Generation (RAG) combines the power of LLMs with external knowledge sources 
to ...
Length: 246 characters



## Module 3: Vector Embeddings & Similarity (15 minutes)
### Converting Text to Vectors

In [10]:
# Initialize embedding model
embeddings = OpenAIEmbeddings()

# Create sample documents
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural language processing helps computers understand human language.",
    "Computer vision enables machines to interpret visual information.",
    "Robotics combines AI with physical systems to create autonomous machines."
]

# Generate embeddings
print("🔢 Generating embeddings...")
doc_embeddings = embeddings.embed_documents(documents)

print(f"✅ Created {len(doc_embeddings)} embeddings")
print(f"📏 Each embedding has {len(doc_embeddings[0])} dimensions")

# Demonstrate similarity search
query = "What is machine learning?"
query_embedding = embeddings.embed_query(query)

# Calculate cosine similarity
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(f"\n🔍 Query: {query}")
print("📊 Similarity Scores:")
for i, doc in enumerate(documents):
    similarity = cosine_similarity(query_embedding, doc_embeddings[i])
    print(f"{similarity:.3f} - {doc}")

🔢 Generating embeddings...
✅ Created 5 embeddings
📏 Each embedding has 1536 dimensions

🔍 Query: What is machine learning?
📊 Similarity Scores:
0.906 - Machine learning is a subset of artificial intelligence.
0.841 - Deep learning uses neural networks with multiple layers.
0.827 - Natural language processing helps computers understand human language.
0.832 - Computer vision enables machines to interpret visual information.
0.841 - Robotics combines AI with physical systems to create autonomous machines.


In [11]:
# Create vector database with Chroma
print("🗄️ Creating vector database...")

# Convert strings to Document objects
docs = [Document(page_content=doc, metadata={"id": i}) for i, doc in enumerate(documents)]

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="ai_knowledge"
)

# Test similarity search
query = "neural networks and deep learning"
similar_docs = vectorstore.similarity_search(query, k=2)

print(f"\n🔍 Query: {query}")
print("📋 Most Similar Documents:")
for i, doc in enumerate(similar_docs):
    print(f"{i+1}. {doc.page_content}")

🗄️ Creating vector database...

🔍 Query: neural networks and deep learning
📋 Most Similar Documents:
1. Deep learning uses neural networks with multiple layers.
2. Machine learning is a subset of artificial intelligence.


## Module 4: RAG System Implementation (25 minutes)
### Building End-to-End RAG

In [12]:
# Create a comprehensive knowledge base
ai_knowledge = """
Artificial Intelligence (AI) Overview:
AI is the simulation of human intelligence in machines that are programmed to think and learn like humans. 
The term may also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

Machine Learning:
Machine learning is a method of data analysis that automates analytical model building. It is a branch of artificial intelligence 
based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention.

Deep Learning:
Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. 
Learning can be supervised, semi-supervised or unsupervised.

Natural Language Processing (NLP):
NLP is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language, 
in particular how to program computers to process and analyze large amounts of natural language data.

Large Language Models (LLMs):
LLMs are a type of artificial intelligence model designed to understand and generate human-like text. They are trained on massive datasets 
containing diverse text from books, articles, websites, and other sources.

Retrieval-Augmented Generation (RAG):
RAG is an AI framework that combines the capabilities of large language models with external knowledge retrieval systems. 
This approach allows AI models to access and incorporate real-time, domain-specific information that wasn't part of their original training data.
"""

print("📚 Knowledge Base Created")
print(f"📏 Total length: {len(ai_knowledge)} characters")

📚 Knowledge Base Created
📏 Total length: 1609 characters


In [13]:
# Process the knowledge base
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ":", ". ", " ", ""]
)

# Split the text
chunks = text_splitter.split_text(ai_knowledge)
print(f"📄 Created {len(chunks)} text chunks")

# Convert to documents
documents = [Document(page_content=chunk, metadata={"chunk_id": i}) for i, chunk in enumerate(chunks)]

# Create new vector store with our knowledge base
knowledge_vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ai_bootcamp_knowledge"
)

print("✅ Vector store created with knowledge base")

📄 Created 5 text chunks
✅ Vector store created with knowledge base


In [14]:
# Create RAG Chain
retriever = knowledge_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Create RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    verbose=True
)

print("🔗 RAG Chain created successfully")

🔗 RAG Chain created successfully


In [ ]:
# class CustomClass(Runnable):
#     pass

In [15]:
# Test RAG System
def test_rag(question: str):
    print(f"❓ Question: {question}")
    print("-" * 50)
    
    result = rag_chain({"query": question})
    
    print(f"🤖 Answer: {result['result']}")
    print("\n📚 Sources:")
    for i, doc in enumerate(result['source_documents']):
        print(f"{i+1}. {doc.page_content[:100]}...")
    print("=" * 70)

# Test with various questions
test_questions = [
    "What is the difference between machine learning and deep learning?",
    "How does RAG work?",
    "What are Large Language Models used for?"
]

for question in test_questions:
    test_rag(question)
    print()

❓ Question: What is the difference between machine learning and deep learning?
--------------------------------------------------


> Entering new RetrievalQA chain...

> Finished chain.
🤖 Answer: The main difference between machine learning and deep learning is that deep learning is a subset of machine learning. Specifically, deep learning is a more advanced form of machine learning that uses artificial neural networks with representation learning to process and understand data. In contrast, machine learning encompasses a broader range of techniques and methods that enable machines to learn from data and make decisions.

📚 Sources:
1. Machine Learning:
Machine learning is a method of data analysis that automates analytical model buil...
2. Artificial Intelligence (AI) Overview:
AI is the simulation of human intelligence in machines that a...
3. Large Language Models (LLMs):
LLMs are a type of artificial intelligence model designed to understan...

❓ Question: How does RAG work?
------

## Module 5: LangChain Framework (20 minutes)
### Chains, Prompts, and Memory

In [16]:
# Custom Prompt Templates
from langchain.prompts import PromptTemplate

# Create a custom prompt for AI education
education_prompt = PromptTemplate(
    input_variables=["topic", "level", "format"],
    template="""
You are an AI education expert. Explain the topic '{topic}' to someone with {level} knowledge level.
Use the following format: {format}

Make the explanation clear, engaging, and appropriate for the audience.

Topic: {topic}
Level: {level}
Format: {format}

Explanation:
"""
)

# Test the prompt
formatted_prompt = education_prompt.format(
    topic="Neural Networks",
    level="beginner",
    format="step-by-step explanation with analogies"
)

response = llm.predict(formatted_prompt)
print("📝 Custom Prompt Response:")
print(response)

📝 Custom Prompt Response:
Step 1: Imagine a neural network as a computer system that is inspired by how our brain works. Just like our brain has interconnected neurons that help us think and learn, a neural network has interconnected nodes that help it process information.

Step 2: Each node in a neural network is like a tiny decision-making unit. These nodes take in input, process it, and produce an output. Just like how our brain's neurons work together to help us make decisions, the nodes in a neural network work together to solve a problem or make a prediction.

Step 3: The connections between nodes in a neural network are like pathways in our brain that help information flow. These connections allow the nodes to communicate with each other and pass on information, which helps the neural network learn and improve over time.

Step 4: When a neural network is trained on a set of data, it adjusts the connections between nodes based on the patterns it finds in the data. This is similar

In [17]:
# Conversation Chain with Memory
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print("💬 Starting Conversation with Memory")
print("=" * 50)

# Simulate a conversation
responses = [
    conversation.predict(input="Hi, I'm learning about AI. Can you help me?"),
    conversation.predict(input="What's the difference between AI and machine learning?"),
    conversation.predict(input="Can you give me an example of the concept you just explained?"),
    conversation.predict(input="What did we talk about first?")
]

for i, response in enumerate(responses, 1):
    print(f"\n🔄 Exchange {i}:")
    print(response)
    print("-" * 30)

💬 Starting Conversation with Memory


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, I'm learning about AI. Can you help me?
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, I'm learning about AI. Can you help me?
AI: Hello! I'd be happy to help you learn about AI. What specific aspect are you interested in learning about? AI is a fascinating field with many different applications and technologies.
Huma

## Module 6: AI Agents & Tools (20 minutes)
### Creating Autonomous AI Agents

In [18]:
# Define custom tools for our AI agent
def calculate_tokens(text: str) -> str:
    """Estimate the number of tokens in a text."""
    # Simple estimation: ~4 characters per token
    estimated_tokens = len(text) // 4
    return f"Estimated tokens: {estimated_tokens} (Text length: {len(text)} characters)"

def ai_concept_explainer(concept: str) -> str:
    """Explain AI concepts with definitions and examples."""
    concepts_db = {
        "llm": "Large Language Model - AI trained on vast text data to understand and generate human-like text",
        "rag": "Retrieval-Augmented Generation - Combines LLM with external knowledge retrieval for better accuracy",
        "embedding": "Vector representation of text that captures semantic meaning for similarity comparisons",
        "agent": "Autonomous AI system that can use tools and make decisions to complete tasks",
        "chain": "Sequence of operations that process input through multiple steps to produce output"
    }
    
    concept_lower = concept.lower()
    if concept_lower in concepts_db:
        return f"Concept: {concept}\nDefinition: {concepts_db[concept_lower]}"
    else:
        return f"Concept '{concept}' not found in database. Available concepts: {', '.join(concepts_db.keys())}"

def search_knowledge_base(query: str) -> str:
    """Search the RAG knowledge base for information."""
    try:
        results = knowledge_vectorstore.similarity_search(query, k=2)
        if results:
            return f"Found information: {results[0].page_content}"
        else:
            return "No relevant information found in knowledge base."
    except:
        return "Error accessing knowledge base."

# Create tools
tools = [
    Tool(
        name="Token Calculator",
        func=calculate_tokens,
        description="Calculate estimated number of tokens in text"
    ),
    Tool(
        name="AI Concept Explainer",
        func=ai_concept_explainer,
        description="Explain AI concepts like LLM, RAG, embedding, agent, chain"
    ),
    Tool(
        name="Knowledge Base Search",
        func=search_knowledge_base,
        description="Search the AI knowledge base for detailed information"
    )
]

print("🛠️ Created 3 custom tools for AI agent")

🛠️ Created 3 custom tools for AI agent


In [19]:
# Initialize AI Agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print("🤖 AI Agent initialized with tools")
print("Available tools:", [tool.name for tool in tools])

🤖 AI Agent initialized with tools
Available tools: ['Token Calculator', 'AI Concept Explainer', 'Knowledge Base Search']


In [20]:
# Test AI Agent
def test_agent(query: str):
    print(f"\n🎯 Agent Task: {query}")
    print("=" * 60)
    try:
        result = agent.run(query)
        print(f"✅ Result: {result}")
    except Exception as e:
        print(f"❌ Error: {e}")
    print("=" * 60)

# Test the agent with different tasks
agent_tasks = [
    "How many tokens would the phrase 'Hello, this is a test of the AI agent system' contain?",
    "Explain what RAG is and then search our knowledge base for more details about it",
    "What is an LLM and how many tokens would a typical explanation contain?"
]

for task in agent_tasks:
    test_agent(task)


🎯 Agent Task: How many tokens would the phrase 'Hello, this is a test of the AI agent system' contain?


> Entering new AgentExecutor chain...
I need to calculate the number of tokens in the given text.
Action: Token Calculator
Action Input: 'Hello, this is a test of the AI agent system'
Observation: Estimated tokens: 11 (Text length: 46 characters)
I have calculated the number of tokens in the text.
Final Answer: The phrase 'Hello, this is a test of the AI agent system' contains 11 tokens.

> Finished chain.
✅ Result: The phrase 'Hello, this is a test of the AI agent system' contains 11 tokens.

🎯 Agent Task: Explain what RAG is and then search our knowledge base for more details about it


> Entering new AgentExecutor chain...
I should first use the AI Concept Explainer to get an explanation of RAG, and then use the Knowledge Base Search to find more detailed information about it.
Action: AI Concept Explainer
Action Input: RAG
Observation: Concept: RAG
Definition: Retrieval-Augmente

## Module 7: Final Project Integration (5 minutes)
### Complete AI Assistant

In [21]:
class AIBootcampAssistant:
    def __init__(self):
        self.llm = ChatOpenAI(temperature=0.7)
        self.memory = ConversationBufferMemory()
        self.vectorstore = knowledge_vectorstore
        self.rag_chain = rag_chain
        self.agent = agent
        
    def chat(self, message: str, mode: str = "conversation") -> str:
        """
        Multi-modal AI assistant that can:
        - conversation: Normal chat with memory
        - rag: Retrieval-augmented generation
        - agent: Use tools to solve tasks
        """
        
        if mode == "rag":
            result = self.rag_chain({"query": message})
            return f"📚 RAG Response: {result['result']}"
            
        elif mode == "agent":
            result = self.agent.run(message)
            return f"🤖 Agent Response: {result}"
            
        else:  # conversation mode
            conversation = ConversationChain(llm=self.llm, memory=self.memory)
            result = conversation.predict(input=message)
            return f"💬 Chat Response: {result}"
    
    def analyze_query(self, query: str) -> str:
        """Analyze what type of response would be best for the query"""
        if any(keyword in query.lower() for keyword in ["explain", "what is", "definition", "concept"]):
            return "rag"
        elif any(keyword in query.lower() for keyword in ["calculate", "count", "search", "find"]):
            return "agent"
        else:
            return "conversation"

# Initialize the complete assistant
assistant = AIBootcampAssistant()
print("🚀 Complete AI Bootcamp Assistant Ready!")

🚀 Complete AI Bootcamp Assistant Ready!


In [22]:
# Demonstration of the complete system
demo_queries = [
    "What is machine learning?",
    "Calculate tokens in 'This is an example sentence for token counting'",
    "Hello! I'm excited to learn about AI today.",
    "Search for information about deep learning in our knowledge base",
    "Can you remember what I said I was excited about?"
]

print("🎬 Complete System Demonstration")
print("=" * 70)

for query in demo_queries:
    # Auto-detect best mode
    suggested_mode = assistant.analyze_query(query)
    
    print(f"\n👤 User: {query}")
    print(f"🧠 Suggested Mode: {suggested_mode}")
    
    try:
        response = assistant.chat(query, mode=suggested_mode)
        print(response)
    except Exception as e:
        print(f"❌ Error: {e}")
    
    print("-" * 50)

🎬 Complete System Demonstration

👤 User: What is machine learning?
🧠 Suggested Mode: rag


> Entering new RetrievalQA chain...

> Finished chain.
📚 RAG Response: Machine learning is a method of data analysis that automates analytical model building. It is a branch of artificial intelligence based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention.
--------------------------------------------------

👤 User: Calculate tokens in 'This is an example sentence for token counting'
🧠 Suggested Mode: agent


> Entering new AgentExecutor chain...
I need to calculate the number of tokens in the given text.
Action: Token Calculator
Action Input: 'This is an example sentence for token counting'
Observation: Estimated tokens: 12 (Text length: 48 characters)
I have the estimated number of tokens in the text.
Final Answer: 12 tokens

> Finished chain.
🤖 Agent Response: 12 tokens
--------------------------------------------------

👤 User: 

## 🎓 Bootcamp Summary & Next Steps

### What You've Built:
1. **LLM Integration** - Direct interaction with language models
2. **Text Processing Pipeline** - Smart chunking and splitting strategies
3. **Vector Database** - Semantic search with embeddings
4. **RAG System** - Knowledge-augmented AI responses
5. **LangChain Framework** - Chains, prompts, and memory management
6. **AI Agents** - Autonomous systems with custom tools
7. **Complete Assistant** - Multi-modal AI system

### Key Concepts Mastered:
- **Prompt Engineering** - Crafting effective AI instructions
- **Semantic Search** - Finding similar content using vector similarity
- **Context Management** - Handling conversation memory and document chunks
- **Tool Integration** - Extending AI capabilities with custom functions
- **System Architecture** - Combining multiple AI components

### Production Considerations:
- **Error Handling** - Robust exception management
- **Rate Limiting** - Managing API costs and quotas
- **Caching** - Storing embeddings and responses
- **Security** - API key management and input validation
- **Scaling** - Async operations and load balancing

### Next Steps:
1. **Deploy** your system using Streamlit or Gradio
2. **Expand** the knowledge base with your domain data
3. **Optimize** for your specific use case
4. **Monitor** performance and user interactions
5. **Iterate** based on feedback and requirements

**Congratulations! You've built a complete AI system in 2 hours! 🎉**

In [ ]:
# Final interactive demo
print("🎮 Interactive Demo - Try your own queries!")
print("Modes: 'rag' for knowledge queries, 'agent' for tool tasks, 'chat' for conversation")
print("Type 'quit' to exit")

while True:
    try:
        user_input = input("\n👤 You: ")
        if user_input.lower() == 'quit':
            print("👋 Thanks for using the AI Bootcamp Assistant!")
            break
            
        # Auto-detect mode or let user specify
        if user_input.startswith('['):
            mode = user_input.split(']')[0][1:]
            query = user_input.split(']')[1].strip()
        else:
            mode = assistant.analyze_query(user_input)
            query = user_input
            
        response = assistant.chat(query, mode=mode)
        print(f"🤖 Assistant ({mode}): {response}")
        
    except KeyboardInterrupt:
        print("\n👋 Session ended!")
        break
    except Exception as e:
        print(f"❌ Error: {e}")

🎮 Interactive Demo - Try your own queries!
Modes: 'rag' for knowledge queries, 'agent' for tool tasks, 'chat' for conversation
Type 'quit' to exit



👤 You:  hi


🤖 Assistant (conversation): 💬 Chat Response: Hello! How can I assist you today?



👤 You:  what can you do?


🤖 Assistant (conversation): 💬 Chat Response: I can do a wide range of tasks, such as answering questions, providing information on various topics, assisting with problem-solving, and even engaging in friendly conversations like the one we're having right now. I have access to a vast amount of data and can use that information to provide accurate and relevant responses. Feel free to ask me anything you'd like to know, and I'll do my best to help!



👤 You:  who is cristiano ronaldo?


🤖 Assistant (conversation): 💬 Chat Response: Cristiano Ronaldo is a professional soccer player from Portugal who is widely considered one of the greatest footballers of all time. He has played for top clubs such as Sporting Lisbon, Manchester United, Real Madrid, and Juventus. Ronaldo has won numerous awards throughout his career, including multiple FIFA Ballon d'Or titles, which are awarded to the best player in the world. He is known for his incredible goal-scoring ability, athleticism, and work ethic on the field. Ronaldo is also active on social media and has a massive following of fans around the world.



👤 You:  do you remember what topics do we discussed?


🤖 Assistant (conversation): 💬 Chat Response: Yes, we discussed AI technology, different types of AI such as machine learning and deep learning, the constant evolution and improvement of AI, Cristiano Ronaldo as a professional soccer player, and the tasks that I can perform as an AI, such as answering questions and providing information on various topics. Is there anything specific you would like to revisit or explore further?
